# 02 Train 2D CNN v0.5

**Purpose:** UI-only tmux control panel for launching new 2D CNN training runs or resuming from existing runs in a selected model directory.  
**Inputs:** `./src/train.py`, `./src/v0.2/training_control_panel.py`, `./src/resume/control_panel.py`, existing model run artifacts under `./models/`.  
**Expected outputs:** run folders under `./models/<model_directory>/runs/<run_id>/` with `train.log`, `best.pt`, `latest.pt`, and metadata files.  
**Artifact write mode:** canonical artifacts are written by `src/train.py`; this notebook orchestrates launch/session/log controls.  
**Decision supported:** `READY_FOR_LAUNCH` vs `PATCH_REQUIRED`


In [1]:
# Repo Setup
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    for parent in repo_root.parents:
        if (parent / 'src').exists() and (parent / 'training-data').exists() and (parent / 'validation-data').exists():
            repo_root = parent
            break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

helper_root = (repo_root / 'src' / 'v0.2').resolve()
if str(helper_root) not in sys.path:
    sys.path.insert(0, str(helper_root))

print(f'repo_root={repo_root}')
print(f'helper_root={helper_root}')


repo_root=/home/mitch/development/raccoon-ball/rb-training
helper_root=/home/mitch/development/raccoon-ball/rb-training/src/v0.2


In [2]:
# Panel Config
REPO_ROOT = repo_root
PYTHON_EXECUTABLE = sys.executable
TRAINING_MODULE = 'src.train'

TRAINING_DATA_ROOT = REPO_ROOT / 'training-data'
VALIDATION_DATA_ROOT = REPO_ROOT / 'validation-data'
MODELS_ROOT = REPO_ROOT / 'models'

MODEL_SUFFIX_DEFAULT = '2d-cnn'
TOPOLOGY_ID_DEFAULT = 'distance_regressor_2d_cnn'
MODEL_NAME_DEFAULT = 'fast_v0_2'
LOG_FILENAME = 'train.log'
DEFAULT_LOG_TAIL_LINES = 180
DEFAULT_LOG_POLL_SECONDS = 5.0
DEFAULT_ADDITIONAL_EPOCHS = 0
NEW_RUN_OPTION_VALUE = '__new_run__'
RESERVED_PARENT_RUN_ID = '[reserved for future implementation]]'

TRAIN_LAUNCH_DEFAULTS = {
    'seed': 42,
    'batch_size': 8,
    'epochs': 32,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'early_stopping_patience': 3,
    'change_note': 'tmux control-panel launch via notebook v0.5',
}

print(f'python={PYTHON_EXECUTABLE}')
print(f'train_module={TRAINING_MODULE}')
print(f'models_root={MODELS_ROOT}')


python=/home/mitch/development/raccoon-ball/.venv/bin/python
train_module=src.train
models_root=/home/mitch/development/raccoon-ball/rb-training/models


In [3]:
# Helper Imports
import html
import json
import re
import threading

import ipywidgets as widgets
from IPython.display import display

import training_control_panel as control
from src.resume import control_panel as resume_control
from src.resume.state import RESUME_STATE_FILENAME, load_resume_state

print(f'loaded_training_helper={control.__file__}')
print(f'loaded_resume_helper={resume_control.__file__}')


loaded_training_helper=/home/mitch/development/raccoon-ball/rb-training/src/v0.2/training_control_panel.py
loaded_resume_helper=/home/mitch/development/raccoon-ball/rb-training/src/resume/control_panel.py


In [4]:
# Control Panel
_RUN_ID_RE = re.compile(r'^run_([0-9]+)$')

status_area = widgets.HTML(value='')
session_suggestion = widgets.HTML(value='')

topology_id_input = widgets.Text(
    value=TOPOLOGY_ID_DEFAULT,
    description='Topology ID',
    layout=widgets.Layout(width='420px'),
)
model_name_input = widgets.Text(
    value=MODEL_NAME_DEFAULT,
    description='Top. Variant',
    layout=widgets.Layout(width='320px'),
)
model_select = widgets.Dropdown(
    options=[],
    value=None,
    description='Model Select',
    layout=widgets.Layout(width='420px'),
)
refresh_models_button = widgets.Button(description='Refresh Models')
get_info_button = widgets.Button(description='Get Info', button_style='info')
model_directory_input = widgets.Text(
    value=control.suggest_model_directory(MODELS_ROOT, model_suffix=MODEL_SUFFIX_DEFAULT),
    description='Model Dir',
    layout=widgets.Layout(width='420px'),
)
run_source_dropdown = widgets.Dropdown(
    options=[('new run', NEW_RUN_OPTION_VALUE)],
    value=NEW_RUN_OPTION_VALUE,
    description='Run Source',
    layout=widgets.Layout(width='560px'),
)

session_name_input = widgets.Text(
    value='',
    description='Session Name',
    placeholder='auto-generated from model name + run id',
    layout=widgets.Layout(width='420px'),
)

seed_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['seed'], description='Seed')
batch_size_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['batch_size'], description='Batch Size')
epochs_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['epochs'], description='Epochs')
additional_epochs_input = widgets.IntText(
    value=DEFAULT_ADDITIONAL_EPOCHS,
    description='Add Epochs',
    layout=widgets.Layout(width='220px'),
)
learning_rate_input = widgets.FloatText(value=TRAIN_LAUNCH_DEFAULTS['learning_rate'], description='LR')
weight_decay_input = widgets.FloatText(value=TRAIN_LAUNCH_DEFAULTS['weight_decay'], description='Weight Decay')
early_stop_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['early_stopping_patience'], description='Early Stop')
enable_lr_scheduler_toggle = widgets.Checkbox(
    value=True,
    description='LR Scheduler',
    indent=False,
    layout=widgets.Layout(width='180px'),
)
change_note_input = widgets.Text(
    value=TRAIN_LAUNCH_DEFAULTS['change_note'],
    description='Change Note',
    layout=widgets.Layout(width='760px'),
)
primary_variable_input = widgets.Text(
    value='',
    description='Primary Var',
    placeholder='e.g. bbox line width 3px -> 1px',
    layout=widgets.Layout(width='760px'),
)
notes_input = widgets.Text(
    value='',
    description='Notes',
    layout=widgets.Layout(width='760px'),
)

run_id_preview = widgets.Text(
    value='',
    description='Run ID',
    disabled=True,
    layout=widgets.Layout(width='320px'),
)
run_register_path = widgets.Text(
    value='',
    description='Run Register',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)
derived_log_path = widgets.Text(
    value='',
    description='Log Path',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)

model_info_output = widgets.Textarea(
    value='',
    description='Model Info',
    layout=widgets.Layout(width='100%', height='220px'),
    disabled=True,
)

session_select = widgets.Dropdown(
    options=[],
    value=None,
    description='Sessions',
    layout=widgets.Layout(width='420px'),
)

refresh_sessions_button = widgets.Button(description='Refresh Sessions')
launch_button = widgets.Button(description='Launch', button_style='success')
end_session_button = widgets.Button(description='End Session', button_style='danger')

log_tail_lines_input = widgets.IntText(
    value=DEFAULT_LOG_TAIL_LINES,
    description='Tail Lines',
    layout=widgets.Layout(width='220px'),
)
poll_interval_input = widgets.FloatText(
    value=DEFAULT_LOG_POLL_SECONDS,
    description='Poll Secs',
    layout=widgets.Layout(width='220px'),
)
auto_refresh_toggle = widgets.ToggleButton(
    value=False,
    description='Auto Refresh',
    icon='refresh',
    layout=widgets.Layout(width='180px'),
)
refresh_log_button = widgets.Button(description='Refresh Log', button_style='info')

log_output = widgets.Textarea(
    value='',
    description='',
    layout=widgets.Layout(width='100%', height='420px'),
    disabled=True,
)

_auto_refresh_state = {'thread': None, 'stop_event': None}
_session_name_state = {'last_auto': ''}
_run_state = {'model_directory': '', 'by_run_id': {}, 'rows': []}


def _run_sort_index(run_id: str) -> int:
    match = _RUN_ID_RE.fullmatch(str(run_id).strip())
    if match:
        return int(match.group(1))
    return 0


def _set_status(message: str, *, is_error: bool = False) -> None:
    color = '#9b1c1c' if is_error else '#0f5132'
    lr_scheduler_text = 'enabled' if bool(enable_lr_scheduler_toggle.value) else 'disabled'
    status_area.value = (
        f"<div style='border:1px solid #d0d7de;padding:8px;border-radius:6px;'>"
        f"<strong style='color:{color};'>Status</strong>"
        f"<div><code>lr_scheduler={lr_scheduler_text}</code></div>"
        f"<div>{html.escape(message)}</div>"
        '</div>'
    )


def _sync_default_session_name(suggested_session: str) -> None:
    current = session_name_input.value.strip()
    last_auto = _session_name_state['last_auto']
    if (not current) or (current == last_auto):
        session_name_input.value = suggested_session
    _session_name_state['last_auto'] = suggested_session


def _load_model_run_rows(model_directory: str) -> tuple[list[dict], Path]:
    model_dir = control.build_model_dir(models_root=MODELS_ROOT, model_directory=model_directory)
    register_path = model_dir / 'run_register.json'
    if not register_path.exists():
        return [], register_path

    with register_path.open('r', encoding='utf-8') as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f'run_register.json must be an object: {register_path}')

    raw_runs = payload.get('runs', [])
    if raw_runs is None:
        raw_runs = []
    if not isinstance(raw_runs, list):
        raise ValueError(f"run_register.json 'runs' must be a list: {register_path}")

    rows: list[dict] = []
    for row in raw_runs:
        if not isinstance(row, dict):
            continue
        run_id = str(row.get('run_id', '')).strip()
        if not _RUN_ID_RE.fullmatch(run_id):
            continue
        rows.append(dict(row))

    rows.sort(key=lambda row: _run_sort_index(str(row.get('run_id', ''))))
    return rows, register_path


def _inspect_run(model_directory: str, row: dict) -> dict:
    run_id = str(row.get('run_id', '')).strip()
    run_dir = control.build_run_dir(
        run_id=run_id,
        models_root=MODELS_ROOT,
        model_directory=model_directory,
    ).resolve()

    info: dict = {
        'run_id': run_id,
        'run_dir': str(run_dir),
        'config_path': str((run_dir / 'config.json').resolve()),
        'resume_state_path': str((run_dir / RESUME_STATE_FILENAME).resolve()),
        'exists': run_dir.exists(),
        'total_epochs': None,
        'completed_epochs': None,
        'remaining_epochs': None,
        'is_complete': None,
        'is_resumable': False,
        'state_error': None,
        'topology_id': str(row.get('topology_id', '')).strip() or topology_id_input.value.strip(),
        'architecture_variant': str(row.get('model_name', '')).strip() or model_name_input.value.strip(),
    }

    config_path = run_dir / 'config.json'
    if config_path.exists():
        try:
            with config_path.open('r', encoding='utf-8') as handle:
                config_payload = json.load(handle)
            total_epochs = config_payload.get('epochs')
            if total_epochs is not None:
                info['total_epochs'] = int(total_epochs)
            topology_id = str(config_payload.get('topology_id', '')).strip()
            if topology_id:
                info['topology_id'] = topology_id
            variant = str(config_payload.get('model_architecture_variant', '')).strip()
            if variant:
                info['architecture_variant'] = variant
        except Exception as exc:
            info['state_error'] = f'config read failed: {exc}'

    resume_state_path = run_dir / RESUME_STATE_FILENAME
    if resume_state_path.exists():
        try:
            resume_state = load_resume_state(resume_state_path, map_location='cpu')
            info['completed_epochs'] = int(resume_state.get('epoch'))
            info['is_resumable'] = True
        except Exception as exc:
            info['state_error'] = f'resume state read failed: {exc}'

    total = info['total_epochs']
    completed = info['completed_epochs']
    if isinstance(total, int) and isinstance(completed, int):
        remaining = max(total - completed, 0)
        info['remaining_epochs'] = remaining
        info['is_complete'] = remaining == 0

    return info


def _build_run_label(info: dict) -> str:
    run_id = info['run_id']
    completed = info.get('completed_epochs')
    total = info.get('total_epochs')
    completed_text = '?' if completed is None else str(int(completed))
    total_text = '?' if total is None else str(int(total))
    label = f'{run_id} | completed {completed_text}/{total_text}'

    remaining = info.get('remaining_epochs')
    if remaining is not None:
        label += f' | remaining {int(remaining)}'
        if int(remaining) == 0:
            label += ' (complete)'
    if not info.get('is_resumable'):
        label += ' | no resume_state'

    state_error = info.get('state_error')
    if state_error:
        label += ' | state warning'
    return label


def _refresh_models(*_):
    try:
        model_dirs = control.list_model_directories(MODELS_ROOT)
    except Exception as exc:
        model_select.options = []
        model_select.value = None
        _set_status(f'Model directory refresh failed: {exc}', is_error=True)
        return

    preferred = model_directory_input.value.strip()
    model_select.options = model_dirs

    if model_dirs:
        if preferred in model_dirs:
            model_select.value = preferred
        else:
            model_select.value = model_dirs[-1]
            model_directory_input.value = model_select.value
    else:
        model_select.value = None


def _refresh_run_options(*_, preferred: str | None = None, show_error: bool = False) -> None:
    model_directory = model_directory_input.value.strip()
    _run_state['model_directory'] = model_directory
    _run_state['rows'] = []
    _run_state['by_run_id'] = {}

    default_options = [('new run', NEW_RUN_OPTION_VALUE)]
    run_source_dropdown.options = default_options
    run_source_dropdown.value = NEW_RUN_OPTION_VALUE

    if not model_directory:
        run_register_path.value = ''
        return

    try:
        rows, register_path = _load_model_run_rows(model_directory)
    except Exception as exc:
        if show_error:
            _set_status(f'Run list refresh failed: {exc}', is_error=True)
        run_register_path.value = ''
        return

    run_register_path.value = str(register_path)
    _run_state['rows'] = rows

    infos = [_inspect_run(model_directory, row) for row in rows]
    by_run_id = {str(info['run_id']): info for info in infos}
    _run_state['by_run_id'] = by_run_id

    options = [('new run', NEW_RUN_OPTION_VALUE)]
    for info in infos:
        options.append((_build_run_label(info), str(info['run_id'])))

    run_source_dropdown.options = options
    desired = preferred if preferred is not None else run_source_dropdown.value
    if desired in by_run_id or desired == NEW_RUN_OPTION_VALUE:
        run_source_dropdown.value = desired
    else:
        run_source_dropdown.value = NEW_RUN_OPTION_VALUE


def _current_source_info() -> dict | None:
    run_value = run_source_dropdown.value
    if run_value in (None, NEW_RUN_OPTION_VALUE):
        return None
    return _run_state['by_run_id'].get(str(run_value))


def _update_preview_paths(show_error: bool = False):
    model_name = model_name_input.value.strip()
    model_directory = model_directory_input.value.strip()
    source_run_value = run_source_dropdown.value

    if not model_directory:
        run_id_preview.value = ''
        derived_log_path.value = ''
        run_register_path.value = ''
        session_suggestion.value = '<em>Suggested session name: <code>rb-&lt;model_name&gt;-&lt;run_id&gt;</code></em>'
        return None

    try:
        next_run_id = control.preview_next_run_id(
            models_root=MODELS_ROOT,
            model_directory=model_directory,
        )
        run_id_preview.value = next_run_id

        log_path = control.build_log_path(
            run_id=next_run_id,
            models_root=MODELS_ROOT,
            model_directory=model_directory,
            log_filename=LOG_FILENAME,
        )
        derived_log_path.value = str(log_path)

        session_base = model_name if model_name else 'model'
        source_info = _current_source_info()
        mode = 'new'
        if source_run_value not in (None, NEW_RUN_OPTION_VALUE) and source_info is not None:
            mode = 'resume'
            suggested_session = f"rb-{session_base}-{source_info['run_id']}-to-{next_run_id}"
        else:
            suggested_session = f'rb-{session_base}-{next_run_id}'

        session_suggestion.value = f"<em>Suggested session name: <code>{html.escape(suggested_session)}</code></em>"
        _sync_default_session_name(suggested_session)

        return {
            'mode': mode,
            'model_name': model_name,
            'model_directory': model_directory,
            'run_id': next_run_id,
            'log_path': str(log_path),
            'suggested_session': suggested_session,
            'source_run_info': source_info,
        }
    except Exception as exc:
        run_id_preview.value = ''
        derived_log_path.value = ''
        if show_error:
            _set_status(f'Path derivation failed: {exc}', is_error=True)
        return None


def _refresh_sessions(*, preferred: str | None = None) -> None:
    try:
        sessions = control.list_sessions()
    except Exception as exc:
        session_select.options = []
        session_select.value = None
        _set_status(f'Session refresh failed: {exc}', is_error=True)
        return

    current = preferred if preferred is not None else session_select.value
    session_select.options = sessions
    if sessions:
        if current in sessions:
            session_select.value = current
        else:
            session_select.value = sessions[0]
    else:
        session_select.value = None


def _resolve_log_path_from_selected_session() -> str | None:
    selected = session_select.value or session_name_input.value.strip()
    if not selected:
        preview = _update_preview_paths(show_error=False)
        return None if preview is None else preview['log_path']

    try:
        resolved = control.resolve_session_run(
            session_name=selected,
            models_root=MODELS_ROOT,
        )
    except Exception as exc:
        _set_status(f'Session lookup failed: {exc}', is_error=True)
        return None

    if resolved is None:
        return None

    model_directory_input.value = resolved['model_directory']
    if resolved['model_directory'] in model_select.options:
        model_select.value = resolved['model_directory']
    run_id_preview.value = resolved['run_id']
    derived_log_path.value = resolved['log_path']
    return str(resolved['log_path'])


def _refresh_log(*_) -> None:
    selected_session = session_select.value or session_name_input.value.strip()
    log_path = _resolve_log_path_from_selected_session()
    if log_path is None:
        if selected_session:
            log_output.value = f'[no registered run for session: {selected_session}]'
        else:
            log_output.value = '[no session selected]'
        return

    try:
        tail_lines = int(log_tail_lines_input.value)
        log_output.value = control.read_log_tail(log_path, max_lines_or_chars=tail_lines)
    except Exception as exc:
        log_output.value = ''
        _set_status(f'Log refresh failed: {exc}', is_error=True)


def _stop_auto_refresh() -> None:
    stop_event = _auto_refresh_state.get('stop_event')
    thread = _auto_refresh_state.get('thread')
    if stop_event is not None:
        stop_event.set()
    if thread is not None and thread.is_alive():
        thread.join(timeout=1.0)
    _auto_refresh_state['thread'] = None
    _auto_refresh_state['stop_event'] = None


def _auto_refresh_loop(stop_event: threading.Event) -> None:
    while not stop_event.is_set():
        try:
            _refresh_log()
        except Exception as exc:
            _set_status(f'Auto refresh failed: {exc}', is_error=True)
        interval = max(0.5, float(poll_interval_input.value))
        stop_event.wait(interval)


def _on_auto_refresh_toggle(change):
    enabled = bool(change['new'])
    if enabled:
        _stop_auto_refresh()
        stop_event = threading.Event()
        thread = threading.Thread(target=_auto_refresh_loop, args=(stop_event,), daemon=True)
        _auto_refresh_state['thread'] = thread
        _auto_refresh_state['stop_event'] = stop_event
        thread.start()
        _set_status('Auto refresh enabled.')
    else:
        _stop_auto_refresh()
        _set_status('Auto refresh disabled.')


def _build_model_info_text(model_directory: str) -> str:
    lines: list[str] = [f'model_directory: {model_directory}', '']
    infos = list(_run_state['by_run_id'].values())
    if not infos:
        lines.append('No registered runs found for this model directory.')
        return "\n".join(lines)

    resumable_count = 0
    incomplete_count = 0
    for info in infos:
        if info.get('is_resumable'):
            resumable_count += 1
        remaining = info.get('remaining_epochs')
        if isinstance(remaining, int) and remaining > 0:
            incomplete_count += 1

    lines.append(f'registered_runs: {len(infos)}')
    lines.append(f'resumable_runs: {resumable_count}')
    lines.append(f'incomplete_runs: {incomplete_count}')
    lines.append('')
    lines.append('Run details (completed / total):')

    for info in infos:
        completed = info.get('completed_epochs')
        total = info.get('total_epochs')
        remaining = info.get('remaining_epochs')
        completed_text = '?' if completed is None else str(int(completed))
        total_text = '?' if total is None else str(int(total))
        remaining_text = '?' if remaining is None else str(int(remaining))
        resume_text = 'yes' if info.get('is_resumable') else 'no'
        lines.append(
            f"- {info['run_id']}: completed={completed_text} / total={total_text}; "
            f"remaining={remaining_text}; resumable={resume_text}"
        )
        state_error = info.get('state_error')
        if state_error:
            lines.append(f'  warning: {state_error}')

    return "\n".join(lines)


def _on_get_info_clicked(_):
    model_directory = model_directory_input.value.strip()
    if not model_directory:
        model_info_output.value = 'Model directory is empty.'
        _set_status('Model directory is required for Get Info.', is_error=True)
        return

    _refresh_run_options(preferred=run_source_dropdown.value, show_error=True)
    _update_preview_paths(show_error=False)
    model_info_output.value = _build_model_info_text(model_directory)
    _set_status(f'Loaded model info for {model_directory}.')


def _on_launch_clicked(_):
    preview = _update_preview_paths(show_error=True)
    if preview is None:
        return

    session_name = session_name_input.value.strip()
    if not session_name:
        _set_status(
            f"Session name is required. Suggested: {preview['suggested_session']}",
            is_error=True,
        )
        return

    if control.session_exists(session_name):
        _set_status(f'Duplicate session name refused: {session_name}', is_error=True)
        return

    try:
        add_epochs = int(additional_epochs_input.value)
        if add_epochs < 0:
            raise ValueError(f'Add Epochs must be >= 0; got {add_epochs}')

        source_info = preview.get('source_run_info')
        is_resume = bool(preview.get('mode') == 'resume' and source_info is not None)
        parent_run_id = RESERVED_PARENT_RUN_ID
        topology_id = topology_id_input.value.strip() or TOPOLOGY_ID_DEFAULT
        model_arch_variant = preview['model_name']

        if is_resume:
            parent_run_id = str(source_info['run_id'])
            source_variant = str(source_info.get('architecture_variant', '')).strip()
            if source_variant:
                model_arch_variant = source_variant
            source_topology_id = str(source_info.get('topology_id', '')).strip()
            if source_topology_id:
                topology_id = source_topology_id

        reservation = control.reserve_run(
            models_root=MODELS_ROOT,
            model_directory=preview['model_directory'],
            model_name=model_arch_variant,
            topology_id=topology_id,
            topology_variant=model_arch_variant,
            session_name=session_name,
            primary_variable_changed=primary_variable_input.value.strip(),
            notes=notes_input.value.strip(),
            parent_run_id=parent_run_id,
        )

        run_id_preview.value = reservation['run_id']
        run_register_path.value = reservation['run_register_path']
        derived_log_path.value = reservation['log_path']

        if not is_resume:
            command = control.build_launch_command(
                run_id=reservation['run_id'],
                model_directory=preview['model_directory'],
                model_architecture_variant=model_arch_variant,
                topology_id=topology_id,
                python_executable=PYTHON_EXECUTABLE,
                training_module=TRAINING_MODULE,
                training_data_root=TRAINING_DATA_ROOT,
                validation_data_root=VALIDATION_DATA_ROOT,
                output_root=MODELS_ROOT,
                seed=int(seed_input.value),
                batch_size=int(batch_size_input.value),
                epochs=int(epochs_input.value),
                learning_rate=float(learning_rate_input.value),
                weight_decay=float(weight_decay_input.value),
                early_stopping_patience=int(early_stop_input.value),
                enable_lr_scheduler=bool(enable_lr_scheduler_toggle.value),
                change_note=change_note_input.value.strip() or 'tmux control-panel launch',
            )
            launch_mode_message = (
                f"Launched new run {reservation['run_id']} in {preview['model_directory']} "
                f'(session {session_name}).'
            )
        else:
            total_epochs = source_info.get('total_epochs')
            completed_epochs = source_info.get('completed_epochs')
            required_remaining = 0
            if isinstance(total_epochs, int) and isinstance(completed_epochs, int):
                required_remaining = max(int(total_epochs) - int(completed_epochs), 0)

            additional_for_resume = required_remaining + add_epochs
            if additional_for_resume <= 0:
                raise ValueError(
                    f"Selected source run {source_info['run_id']} appears complete "
                    f"({completed_epochs}/{total_epochs}). Set Add Epochs > 0 or choose new run."
                )

            command = resume_control.build_resume_launch_command(
                run_id=reservation['run_id'],
                model_directory=preview['model_directory'],
                source_run_dir=source_info['run_dir'],
                additional_epochs=int(additional_for_resume),
                python_executable=PYTHON_EXECUTABLE,
                training_module=TRAINING_MODULE,
                output_root=MODELS_ROOT,
                change_note=change_note_input.value.strip() or 'resume launch',
            )
            launch_mode_message = (
                f"Launched resume run {source_info['run_id']} -> {reservation['run_id']} "
                f"(required_remaining={required_remaining}, add_epochs={add_epochs}, "
                f"total_additional={additional_for_resume}; session {session_name})."
            )

        control.launch_session(
            session_name=session_name,
            command=command,
            log_path=reservation['log_path'],
            working_directory=REPO_ROOT,
        )

        _refresh_run_options(preferred=run_source_dropdown.value, show_error=False)
        _refresh_sessions(preferred=session_name)
        _refresh_log()
        model_info_output.value = _build_model_info_text(preview['model_directory'])
        _set_status(launch_mode_message)
    except Exception as exc:
        _set_status(f'Launch failed: {exc}', is_error=True)


def _on_end_session_clicked(_):
    selected_session = session_select.value
    if not selected_session:
        _set_status('Select a session to end.', is_error=True)
        return
    try:
        ended = control.end_session(selected_session)
        if ended:
            _set_status(f'Ended session {selected_session}.')
        else:
            _set_status(f'Session not found: {selected_session}', is_error=True)
        _refresh_sessions()
        _refresh_log()
    except Exception as exc:
        _set_status(f'End session failed: {exc}', is_error=True)


def _on_session_changed(change):
    selected = change['new']
    if selected:
        session_name_input.value = selected
    _refresh_log()


def _on_preview_inputs_changed(_):
    _refresh_run_options(preferred=run_source_dropdown.value, show_error=False)
    _update_preview_paths(show_error=False)


def _on_model_selected(change):
    selected = change['new']
    if selected:
        model_directory_input.value = str(selected)
    _refresh_run_options(preferred=run_source_dropdown.value, show_error=False)
    _update_preview_paths(show_error=False)


def _on_run_source_changed(_):
    _update_preview_paths(show_error=False)


launch_button.on_click(_on_launch_clicked)
end_session_button.on_click(_on_end_session_clicked)
refresh_sessions_button.on_click(lambda _: _refresh_sessions())
refresh_log_button.on_click(_refresh_log)
refresh_models_button.on_click(_refresh_models)
get_info_button.on_click(_on_get_info_clicked)
session_select.observe(_on_session_changed, names='value')
topology_id_input.observe(_on_preview_inputs_changed, names='value')
model_name_input.observe(_on_preview_inputs_changed, names='value')
model_directory_input.observe(_on_preview_inputs_changed, names='value')
model_select.observe(_on_model_selected, names='value')
run_source_dropdown.observe(_on_run_source_changed, names='value')
auto_refresh_toggle.observe(_on_auto_refresh_toggle, names='value')

_refresh_models()
_refresh_run_options(preferred=NEW_RUN_OPTION_VALUE, show_error=False)
_update_preview_paths(show_error=False)
_refresh_sessions()
model_info_output.value = 'Click Get Info to load completed/total epochs for runs in the selected model directory.'
_set_status('Control panel ready.')

display(widgets.VBox([
    status_area,
    widgets.HBox([topology_id_input, model_name_input, model_directory_input]),
    widgets.HBox([model_select, refresh_models_button, get_info_button]),
    run_source_dropdown,
    model_info_output,
    session_name_input,
    session_suggestion,
    widgets.HBox([seed_input, batch_size_input, epochs_input, additional_epochs_input]),
    widgets.HBox([learning_rate_input, weight_decay_input, early_stop_input, enable_lr_scheduler_toggle]),
    change_note_input,
    primary_variable_input,
    notes_input,
    widgets.HBox([run_id_preview]),
    run_register_path,
    derived_log_path,
    widgets.HBox([launch_button, refresh_sessions_button, end_session_button]),
    session_select,
    widgets.HBox([refresh_log_button, log_tail_lines_input, poll_interval_input, auto_refresh_toggle]),
    log_output,
]))


## Final Verdict

`READY_FOR_LAUNCH` when the panel can select a model, enumerate runs with `new run` first, fetch model/run epoch info via `Get Info`, and launch either fresh training or resume training with `Add Epochs` support.
